In [ ]:
%pip install yfinance

import yfinance as yf

ticker = yf.Ticker("aapl")

price = ticker.history(start="2009-02-14", end="2020-06-11")
print(price)

                                Open       High        Low      Close  \
Date                                                                    
2009-02-17 00:00:00-05:00   2.902517   2.907610   2.824913   2.832403   
2009-02-18 00:00:00-05:00   2.847984   2.871954   2.778171   2.827609   
2009-02-19 00:00:00-05:00   2.797645   2.824012   2.699966   2.715847   
2009-02-20 00:00:00-05:00   2.678692   2.768581   2.666707   2.732626   
2009-02-23 00:00:00-05:00   2.746109   2.756596   2.592100   2.605283   
...                              ...        ...        ...        ...   
2020-06-04 00:00:00-04:00  78.593211  78.891211  77.718577  78.091690   
2020-06-05 00:00:00-04:00  78.341231  80.376381  78.312158  80.315811   
2020-06-08 00:00:00-04:00  80.012978  80.824618  79.303100  80.790695   
2020-06-09 00:00:00-04:00  80.470869  83.734373  80.439372  83.341881   
2020-06-10 00:00:00-04:00  84.289205  85.953667  83.850679  85.486069   

                              Volume  Dividends  S

In [13]:
#Collect Features, we want them a day behind, we give yesturdays stats to guess todays price, so score follows current day
price["Change"] = price["Close"].shift(1) - price["Open"].shift(1)
price["Score"] = (price["Close"] > price["Open"]).astype(int)
price["%Change"] = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
price["YestScore"] = price["Score"].shift(1)
price["5DayMean"] = price["Close"].rolling(5).mean().shift(1)
price["5DayVoli"] = price["Close"].rolling(5).std().shift(1)
price["5DayPerf"] = price["Score"].rolling(5).sum().shift(1)

price.dropna(inplace=True)

print(price["5DayPerf"])

Date
2009-02-24 00:00:00-05:00    1.0
2009-02-25 00:00:00-05:00    2.0
2009-02-26 00:00:00-05:00    3.0
2009-02-27 00:00:00-05:00    3.0
2009-03-02 00:00:00-05:00    3.0
                            ... 
2020-06-04 00:00:00-04:00    4.0
2020-06-05 00:00:00-04:00    3.0
2020-06-08 00:00:00-04:00    4.0
2020-06-09 00:00:00-04:00    4.0
2020-06-10 00:00:00-04:00    4.0
Name: 5DayPerf, Length: 2844, dtype: float64


In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

features = ["Change", "%Change", "CloseToOpen", "YestScore", "5DayMean", "5DayVoli", "5DayPerf"]

priceTrain = price[price.index <= "2018-06-11"]
priceTest = price[price.index > "2018-06-11"]

results = []

for i in range(len(priceTest)):
    xTrain = priceTrain[features]
    yTrain = priceTrain["Score"]

    scaler = StandardScaler()
    xTrain = scaler.fit_transform(xTrain)

    model = LogisticRegression()
    model.fit(xTrain, yTrain)

    #Grab yesturdays information to predict todays, data already transformed so i is yesturday
    yest = priceTest.iloc[[i]]
    yestScal = scaler.transform(yest[features])
    pred = model.predict(yestScal)[0]
    prob = model.predict_proba(yestScal)[0][1]
    print("Analyzing Day: ", priceTest.index[i])

    results.append({"Date": priceTest.index[i],
                    "Score": priceTest["Score"].iloc[i],
                    "Prediction": pred,
                    "Probability": prob,
                    "Open": priceTest["Open"].iloc[i],
                    "Close": priceTest["Close"].iloc[i]})
    
    priceTrain = pd.concat([priceTrain, yest])

Analyzing Day:  2018-06-12 00:00:00-04:00
Analyzing Day:  2018-06-13 00:00:00-04:00
Analyzing Day:  2018-06-14 00:00:00-04:00
Analyzing Day:  2018-06-15 00:00:00-04:00
Analyzing Day:  2018-06-18 00:00:00-04:00
Analyzing Day:  2018-06-19 00:00:00-04:00
Analyzing Day:  2018-06-20 00:00:00-04:00
Analyzing Day:  2018-06-21 00:00:00-04:00
Analyzing Day:  2018-06-22 00:00:00-04:00
Analyzing Day:  2018-06-25 00:00:00-04:00
Analyzing Day:  2018-06-26 00:00:00-04:00
Analyzing Day:  2018-06-27 00:00:00-04:00
Analyzing Day:  2018-06-28 00:00:00-04:00
Analyzing Day:  2018-06-29 00:00:00-04:00
Analyzing Day:  2018-07-02 00:00:00-04:00
Analyzing Day:  2018-07-03 00:00:00-04:00
Analyzing Day:  2018-07-05 00:00:00-04:00
Analyzing Day:  2018-07-06 00:00:00-04:00
Analyzing Day:  2018-07-09 00:00:00-04:00
Analyzing Day:  2018-07-10 00:00:00-04:00
Analyzing Day:  2018-07-11 00:00:00-04:00
Analyzing Day:  2018-07-12 00:00:00-04:00
Analyzing Day:  2018-07-13 00:00:00-04:00
Analyzing Day:  2018-07-16 00:00:0

In [15]:
results = pd.DataFrame(results)

score = results["Score"]
pred = results["Prediction"]
op = results["Open"]
close = results["Close"]

accuracy = accuracy_score(score, pred)
precision = precision_score(score, pred, zero_division=0)
recall = recall_score(score, pred, zero_division=0)
f1 = f1_score(score, pred, zero_division=0)

print("Accuracy: ", accuracy, "\nPrecision: ", precision, "\nRecall: ", recall, "\nF1: ", f1)



Accuracy:  0.5288270377733598 
Precision:  0.5602409638554217 
Recall:  0.6714801444043321 
F1:  0.6108374384236454


In [16]:
#Find out how much you would have made/lost with this method
totalIn = 0
totalOut = 0
totalInvested = 0
totalStoodOut = 0
alltradein = 0
alltradeout = 0
perftradein = 0
perftradeout = 0
for i, o, c, s in zip(pred.values, op.values, close.values, score.values):
    if i == 1:
        totalIn += 100
        totalInvested += 1
        temp = ((c - o)/o) * 100
        totalOut += 100 + temp
        print(temp, o, c)
    else:
        totalStoodOut += 1
    alltradein += 100
    alltradeout += (((c-o)/o) * 100) + 100
    if s == 1:
        perftradein += 100
        perftradeout += (((c-o)/o) * 100) + 100

print(totalIn)
print(totalOut)
print(totalInvested)
print(totalStoodOut)
print(alltradein)
print(alltradeout)
print(perftradein)
print(perftradeout)

-0.8938786181636795 45.37893351770335 44.97330093383789
-0.391542671913891 45.17376942021792 44.99689483642578
0.08049041672747455 43.947433686908205 43.98280715942383
-0.9559376694968249 44.159687919681474 43.737548828125
-0.64474370283893 43.893201441575606 43.61020278930664
-0.670662905383354 43.25173179889428 42.9616584777832
0.07556914271810804 43.69037223212883 43.723388671875
1.3752578346130506 43.72811255310879 44.32948684692383
-0.18876860032398937 44.97565843531915 44.8907585144043
-0.3289098764920371 44.45446172771265 44.30824661254883
-0.3185049064032872 45.16669673769916 45.0228385925293
1.154516533934222 44.735120184452185 45.25159454345703
0.9116387588738232 45.5298744916329 45.944942474365234
-1.0369933807545815 45.256299767040694 44.78699493408203
-0.005259992361791533 44.87896661776497 44.87660598754883
0.514426598182083 49.053222058835125 49.305564880371094
-1.0558029029896252 49.36452244431482 48.84333038330078
0.5823814270659097 48.59334036963655 48.876338958740234